# SIREN — Assignment 3: Inference Notebook
## Experimentation & Expansion
**Group ID09 | FAST-NUCES | Dr. Zohair Ahmed | April 2026**

---

This notebook reproduces **all A3 experiments** from scratch in a single, self-contained run.
It mirrors the structure of the A2 inference notebook and extends it with the four A3 improvements.

| Section | Experiment | Key Hyperparams |
|---|---|---|
| **0** | Setup & Model Definitions | — |
| **1** | A2 Baseline Repro — Image Fitting | 5×256, ω₀=30, 10K steps |
| **2** | Improvement I — CAOS-SIREN (Valeena) | ω₀ annealed 10→60, 25→45 |
| **3** | Improvement II — Fourier Frequency Loss (Valeena) | λ ∈ {0.01, 0.05, 0.1} |
| **4** | Improvement III — H-SIREN + SDF (Maryam) | sin(sinh(2x)) first layer |
| **5** | Improvement IV-A — Helmholtz k-Ablation (Laiba) | k ∈ {1,5,20,50}, 1K steps |
| **6** | Improvement IV-B — Wave Equation IC Study (Laiba) | 3 ICs × 500 steps |
| **7** | Bonus — MRI Super-Resolution (Laiba) | IXI T1, 2×/4×, 400 steps |
| **8** | Summary Tables & Consolidated Results | — |

> **Runtime tip:** Enable GPU under *Runtime → Change runtime type → T4 GPU* before running.

---
# Section 0 — Setup

In [ ]:
# ── Google Drive (optional – comment out if running locally) ──
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    DRIVE_ROOT = '/content/drive/MyDrive/SIREN_A3_Inference'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'Drive mounted. Outputs → {DRIVE_ROOT}')
except ImportError:
    import os
    DRIVE_ROOT = './siren_a3_outputs'
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    print(f'Local mode. Outputs → {DRIVE_ROOT}')

In [ ]:
# ── Clone official SIREN repo & apply compatibility patches ──
import os
if not os.path.exists('/content/siren'):
    os.system('git clone https://github.com/vsitzmann/siren.git /content/siren -q')
    # Patch 1: renamed class in newer torchvision
    os.system("sed -i 's/NeuralProcessImplicit2DHypernetBVP/NeuralProcessImplicit2DHypernet/g' /content/siren/utils.py")
    # Patch 2: deprecated skimage API
    os.system("sed -i 's/from skimage.measure import compare_psnr, compare_ssim/from skimage.metrics import peak_signal_noise_ratio as compare_psnr, structural_similarity as compare_ssim/g' /content/siren/utils.py")

# ── Install extra packages ──
os.system('pip install configargparse cmapy scikit-video scikit-image torchio -q')
print('Packages ready.')

In [ ]:
import sys, time, csv, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/content/siren')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from collections import OrderedDict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import skimage
from PIL import Image
from torchvision.transforms import Resize, Compose, ToTensor, Normalize
import scipy.ndimage
import scipy.io.wavfile as wavfile
from skimage.transform import resize as sk_resize
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB')

## 0.1 — Model Definitions
All models used across A3 experiments are defined here once.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  SIREN — Standard (used in A2 baseline and all A3 experiments)
# ═══════════════════════════════════════════════════════════════
class SineLayer(nn.Module):
    """Single SIREN layer: sin(ω₀ · Wx + b)
    See Sitzmann et al. 2020, Section 3.2.
    """
    def __init__(self, in_features, out_features, bias=True,
                 is_first=False, omega_0=30.):
        super().__init__()
        self.omega_0 = omega_0
        self.is_first = is_first
        self.in_features = in_features
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        self._init_weights()

    def _init_weights(self):
        with torch.no_grad():
            if self.is_first:
                self.linear.weight.uniform_(-1 / self.in_features,
                                             1 / self.in_features)
            else:
                self.linear.weight.uniform_(
                    -np.sqrt(6 / self.in_features) / self.omega_0,
                     np.sqrt(6 / self.in_features) / self.omega_0)

    def forward(self, x):
        return torch.sin(self.omega_0 * self.linear(x))

    def forward_with_intermediate(self, x):
        intermediate = self.omega_0 * self.linear(x)
        return torch.sin(intermediate), intermediate


class Siren(nn.Module):
    """Full SIREN MLP.  Default: 5 hidden layers × 256 units, ω₀=30."""
    def __init__(self, in_features=2, hidden_features=256, hidden_layers=4,
                 out_features=1, outermost_linear=True,
                 first_omega_0=30., hidden_omega_0=30.):
        super().__init__()
        layers = [SineLayer(in_features, hidden_features,
                            is_first=True, omega_0=first_omega_0)]
        for _ in range(hidden_layers):
            layers.append(SineLayer(hidden_features, hidden_features,
                                    omega_0=hidden_omega_0))
        if outermost_linear:
            final = nn.Linear(hidden_features, out_features)
            with torch.no_grad():
                final.weight.uniform_(
                    -np.sqrt(6 / hidden_features) / hidden_omega_0,
                     np.sqrt(6 / hidden_features) / hidden_omega_0)
            layers.append(final)
        else:
            layers.append(SineLayer(hidden_features, out_features,
                                    omega_0=hidden_omega_0))
        self.net = nn.Sequential(*layers)

    def forward(self, coords):
        coords = coords.clone().detach().requires_grad_(True)
        output = self.net(coords)
        return output, coords

    def forward_simple(self, x):
        """No grad bookkeeping — used for MRI / Helmholtz / Wave."""
        return self.net(x)


# ═══════════════════════════════════════════════════════════════
#  H-SIREN  (Improvement III — Maryam)  sin(sinh(2x)) first layer
# ═══════════════════════════════════════════════════════════════
class HSineLayer(nn.Module):
    """H-SIREN first layer: sin(sinh(2x)).  Gao & Jaiman 2024."""
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features, bias=bias)
        with torch.no_grad():
            self.linear.weight.uniform_(-1 / in_features, 1 / in_features)

    def forward(self, x):
        return torch.sin(torch.sinh(2.0 * self.linear(x)))


class HSiren(nn.Module):
    """SIREN with H-SIREN first-layer activation."""
    def __init__(self, in_features=2, hidden_features=256, hidden_layers=4,
                 out_features=1, outermost_linear=True, omega_0=30.):
        super().__init__()
        layers = [HSineLayer(in_features, hidden_features)]
        for _ in range(hidden_layers):
            layers.append(SineLayer(hidden_features, hidden_features,
                                    omega_0=omega_0))
        if outermost_linear:
            final = nn.Linear(hidden_features, out_features)
            with torch.no_grad():
                final.weight.uniform_(
                    -np.sqrt(6 / hidden_features) / omega_0,
                     np.sqrt(6 / hidden_features) / omega_0)
            layers.append(final)
        else:
            layers.append(SineLayer(hidden_features, out_features, omega_0=omega_0))
        self.net = nn.Sequential(*layers)

    def forward(self, coords):
        coords = coords.clone().detach().requires_grad_(True)
        return self.net(coords), coords

    def forward_simple(self, x):
        return self.net(x)


# ═══════════════════════════════════════════════════════════════
#  ReLU MLP  (baseline for MRI SR and image fitting)
# ═══════════════════════════════════════════════════════════════
class ReLUMLP(nn.Module):
    def __init__(self, in_features=2, hidden_features=256,
                 hidden_layers=4, out_features=1):
        super().__init__()
        ls = [nn.Linear(in_features, hidden_features), nn.ReLU()]
        for _ in range(hidden_layers):
            ls += [nn.Linear(hidden_features, hidden_features), nn.ReLU()]
        ls.append(nn.Linear(hidden_features, out_features))
        self.net = nn.Sequential(*ls)

    def forward(self, x):
        return self.net(x)


# ═══════════════════════════════════════════════════════════════
#  Differential operators  (used in A2 and Improvement III)
# ═══════════════════════════════════════════════════════════════
def gradient(y, x, grad_outputs=None):
    if grad_outputs is None:
        grad_outputs = torch.ones_like(y)
    return torch.autograd.grad(y, [x], grad_outputs=grad_outputs,
                                create_graph=True)[0]

def divergence(y, x):
    div = 0.
    for i in range(y.shape[-1]):
        div += torch.autograd.grad(y[..., i], x,
                                    torch.ones_like(y[..., i]),
                                    create_graph=True)[0][..., i:i+1]
    return div

def laplace(y, x):
    return divergence(gradient(y, x), x)


# ═══════════════════════════════════════════════════════════════
#  Utility helpers
# ═══════════════════════════════════════════════════════════════
def get_mgrid(sidelen, dim=2):
    """Flattened coordinate grid in [-1, 1]^dim."""
    tensors = tuple(dim * [torch.linspace(-1, 1, steps=sidelen)])
    mgrid = torch.stack(torch.meshgrid(*tensors, indexing='ij'), dim=-1)
    return mgrid.reshape(-1, dim)

def psnr(pred, gt):
    mse = ((pred - gt) ** 2).mean().item()
    return 10 * np.log10(1.0 / (mse + 1e-10))

def make_grid_2d(sz):
    lin = torch.linspace(-1, 1, sz)
    gy, gx = torch.meshgrid(lin, lin, indexing='ij')
    return torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=-1)

def save_fig(name):
    path = os.path.join(DRIVE_ROOT, name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  Saved → {path}')

print('All model definitions loaded ✓')

---
# Section 1 — A2 Baseline: Image Fitting
Reproduces the A2 cameraman image-fitting result to confirm the baseline PSNR ≈ 40.37 dB.
Architecture: 5×256, ω₀=30, Adam lr=1e-4, 10 000 steps.

In [ ]:
def get_cameraman_tensor(sidelength):
    img = Image.fromarray(skimage.data.camera())
    transform = Compose([
        Resize(sidelength),
        ToTensor(),
        Normalize(torch.Tensor([0.5]), torch.Tensor([0.5]))
    ])
    return transform(img)


class ImageFitting(Dataset):
    def __init__(self, sidelength):
        super().__init__()
        img = get_cameraman_tensor(sidelength)
        self.pixels = img.permute(1, 2, 0).view(-1, 1)
        self.coords = get_mgrid(sidelength, 2)

    def __len__(self): return 1

    def __getitem__(self, idx):
        if idx > 0: raise IndexError
        return self.coords, self.pixels


SIDELENGTH  = 256
A2_STEPS    = 10_000
LOG_EVERY   = 1_000

cameraman   = ImageFitting(SIDELENGTH)
dataloader  = DataLoader(cameraman, batch_size=1, pin_memory=True, num_workers=0)

img_siren   = Siren(in_features=2, out_features=1, hidden_features=256,
                    hidden_layers=4, outermost_linear=True).to(device)
optim       = torch.optim.Adam(img_siren.parameters(), lr=1e-4)

model_input, ground_truth = next(iter(dataloader))
model_input, ground_truth = model_input.to(device), ground_truth.to(device)

a2_losses, a2_psnr_curve = [], []
print('=== A2 Baseline: Image Fitting (10 000 steps) ===')
t0 = time.time()
for step in range(A2_STEPS):
    model_output, coords = img_siren(model_input)
    loss = ((model_output - ground_truth) ** 2).mean()
    optim.zero_grad()
    loss.backward()
    optim.step()
    a2_losses.append(loss.item())
    a2_psnr_curve.append(psnr(model_output.detach(), ground_truth))
    if step % LOG_EVERY == 0:
        print(f'  Step {step:5d} | Loss {loss.item():.6f} | PSNR {a2_psnr_curve[-1]:.2f} dB')

A2_BASELINE_PSNR = a2_psnr_curve[-1]
print(f'\n  ✓ Final PSNR = {A2_BASELINE_PSNR:.2f} dB  (paper reports ~40.37 dB)')
print(f'  Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
img_grad      = gradient(model_output, coords)
img_laplacian = laplace(model_output, coords)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(ground_truth.cpu().view(SIDELENGTH, SIDELENGTH).detach().numpy(), cmap='gray')
axes[0].set_title('Ground Truth', fontweight='bold')
axes[1].imshow(model_output.cpu().view(SIDELENGTH, SIDELENGTH).detach().numpy(), cmap='gray')
axes[1].set_title(f'SIREN Output\nPSNR={A2_BASELINE_PSNR:.2f} dB', fontweight='bold')
axes[2].imshow(img_grad.norm(dim=-1).cpu().view(SIDELENGTH, SIDELENGTH).detach().numpy())
axes[2].set_title('Gradient Magnitude', fontweight='bold')
axes[3].imshow(img_laplacian.cpu().view(SIDELENGTH, SIDELENGTH).detach().numpy())
axes[3].set_title('Laplacian', fontweight='bold')
for ax in axes: ax.axis('off')
plt.suptitle('A2 Baseline — Image Fitting (Cameraman)', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('a2_baseline_image.png')

---
# Section 2 — Improvement I: CAOS-SIREN (Valeena Afzal)
**Cosine Annealed Omega Schedule (CAOS).**  
ω₀ is annealed via cosine schedule over 10 000 training steps.  
Two ranges tested: narrow (25→45) and wide (10→60).  
Hypothesis: spending more time at low ω₀ early captures global structure first.

Expected outcome (from report): CAOS destabilises the SIREN init scheme → both runs fail vs baseline.

In [ ]:
def caos_omega(step, total_steps, omega_low, omega_high):
    """Cosine-annealed ω₀ schedule: low → high."""
    return omega_low + (omega_high - omega_low) * \
           (1 - np.cos(np.pi * step / total_steps)) / 2

def train_caos(omega_low, omega_high, total_steps=10_000,
               sidelength=256, lr=1e-4, log_every=1000):
    ds  = ImageFitting(sidelength)
    dl  = DataLoader(ds, batch_size=1, pin_memory=True, num_workers=0)
    mi, gt = next(iter(dl))
    mi, gt = mi.to(device), gt.to(device)

    # Initialise at omega_low
    model = Siren(in_features=2, out_features=1, hidden_features=256,
                  hidden_layers=4, outermost_linear=True,
                  first_omega_0=omega_low, hidden_omega_0=omega_low).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    losses, psnr_curve = [], []
    label = f'CAOS {omega_low}→{omega_high}'
    print(f'  Training {label} ...')
    t0 = time.time()
    for step in range(total_steps):
        # Update ω₀ in all SineLayers
        w = caos_omega(step, total_steps, omega_low, omega_high)
        for m in model.net.modules():
            if isinstance(m, SineLayer):
                m.omega_0 = w

        out, _ = model(mi)
        loss   = ((out - gt) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        psnr_curve.append(psnr(out.detach(), gt))
        if step % log_every == 0:
            print(f'    Step {step:5d} | ω₀={w:.1f} | Loss {loss.item():.6f} | PSNR {psnr_curve[-1]:.2f} dB')

    elapsed = (time.time() - t0) / 60
    final_psnr = psnr_curve[-1]
    print(f'  Done {elapsed:.1f}m | Final PSNR = {final_psnr:.2f} dB\n')
    with torch.no_grad():
        pred, _ = model(mi)
    return {'losses': losses, 'psnr': psnr_curve, 'final_psnr': final_psnr,
            'pred': pred.cpu().view(sidelength, sidelength).numpy(),
            'label': label}


CAOS_STEPS = 10_000
print('=== Improvement I: CAOS-SIREN ===')
caos_narrow = train_caos(25, 45, CAOS_STEPS)
caos_wide   = train_caos(10, 60, CAOS_STEPS)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Loss curves
axes[0].semilogy(caos_narrow['losses'], label=caos_narrow['label'], color='orange')
axes[0].semilogy(caos_wide['losses'],   label=caos_wide['label'],   color='red')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title('CAOS-SIREN — Training Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# PSNR curves
axes[1].plot(caos_narrow['psnr'], label=caos_narrow['label'], color='orange')
axes[1].plot(caos_wide['psnr'],   label=caos_wide['label'],   color='red')
axes[1].axhline(A2_BASELINE_PSNR, color='blue', ls='--', label=f'A2 Baseline ({A2_BASELINE_PSNR:.1f} dB)')
axes[1].set_xlabel('Step'); axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('CAOS-SIREN — PSNR Curve', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Bar chart comparison
models = ['A2 Baseline', caos_narrow['label'], caos_wide['label']]
psnrs  = [A2_BASELINE_PSNR, caos_narrow['final_psnr'], caos_wide['final_psnr']]
colors = ['steelblue', 'orange', 'red']
bars   = axes[2].bar(models, psnrs, color=colors, edgecolor='black', lw=0.8)
for b, v in zip(bars, psnrs):
    axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                 f'{v:.2f} dB', ha='center', va='bottom', fontweight='bold')
axes[2].set_ylabel('Final PSNR (dB)'); axes[2].set_ylim(0, max(psnrs)*1.15)
axes[2].set_title('PSNR Comparison vs A2 Baseline', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Improvement I — CAOS-SIREN Results (Valeena Afzal)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp1_caos_siren.png')

print(f'\n  Baseline PSNR : {A2_BASELINE_PSNR:.2f} dB')
print(f'  CAOS 25→45    : {caos_narrow["final_psnr"]:.2f} dB  (Δ {caos_narrow["final_psnr"]-A2_BASELINE_PSNR:.2f} dB)')
print(f'  CAOS 10→60    : {caos_wide["final_psnr"]:.2f} dB  (Δ {caos_wide["final_psnr"]-A2_BASELINE_PSNR:.2f} dB)')

---
# Section 3 — Improvement II: Fourier Frequency Loss (Valeena Afzal)
**Combined loss:** `L_total = L2 + λ · ‖|FFT(pred)| - |FFT(gt)|‖₁`  
Three lambda values: λ ∈ {0.01, 0.05, 0.1}.  
Hypothesis: penalising spectral differences improves high-frequency detail.  
Expected outcome: FFT loss conflicts with SIREN's natural frequency learning → PSNR drops.

In [ ]:
def fourier_frequency_loss(pred, gt):
    """L1 loss in the magnitude spectrum."""
    fft_pred = torch.fft.fft2(pred)
    fft_gt   = torch.fft.fft2(gt)
    return torch.mean(torch.abs(torch.abs(fft_pred) - torch.abs(fft_gt)))


def train_ffl(lambda_ffl, total_steps=10_000, sidelength=256,
              lr=1e-4, log_every=1000):
    ds  = ImageFitting(sidelength)
    dl  = DataLoader(ds, batch_size=1, pin_memory=True, num_workers=0)
    mi, gt = next(iter(dl))
    mi, gt = mi.to(device), gt.to(device)

    model = Siren(in_features=2, out_features=1, hidden_features=256,
                  hidden_layers=4, outermost_linear=True).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    losses_total, losses_l2, psnr_curve = [], [], []
    label = f'FFL λ={lambda_ffl}'
    print(f'  Training {label} ...')
    t0 = time.time()
    for step in range(total_steps):
        out, _ = model(mi)
        l2  = ((out - gt) ** 2).mean()
        gt_2d  = gt.view(sidelength, sidelength)
        out_2d = out.view(sidelength, sidelength)
        ffl = fourier_frequency_loss(out_2d, gt_2d)
        loss = l2 + lambda_ffl * ffl
        opt.zero_grad(); loss.backward(); opt.step()
        losses_total.append(loss.item())
        losses_l2.append(l2.item())
        psnr_curve.append(psnr(out.detach(), gt))
        if step % log_every == 0:
            print(f'    Step {step:5d} | Total {loss.item():.4f} | L2 {l2.item():.6f} | PSNR {psnr_curve[-1]:.2f} dB')

    elapsed = (time.time() - t0) / 60
    final_psnr = psnr_curve[-1]
    print(f'  Done {elapsed:.1f}m | Final PSNR = {final_psnr:.2f} dB\n')
    return {'losses_total': losses_total, 'losses_l2': losses_l2,
            'psnr': psnr_curve, 'final_psnr': final_psnr, 'label': label}


LAMBDA_VALUES = [0.01, 0.05, 0.1]
FFL_STEPS     = 10_000
print('=== Improvement II: Fourier Frequency Loss ===')
ffl_results = {lam: train_ffl(lam, FFL_STEPS) for lam in LAMBDA_VALUES}

In [ ]:
colors_ffl = {'0.01': 'blue', '0.05': 'red', '0.1': 'cyan'}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Total loss curves
for lam in LAMBDA_VALUES:
    r = ffl_results[lam]
    axes[0].semilogy(r['losses_total'], label=r['label'], lw=2)
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss (log)')
axes[0].set_title('FFL — Total Loss Curves', fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# PSNR curves
axes[1].axhline(A2_BASELINE_PSNR, color='black', ls='--',
                label=f'Baseline ({A2_BASELINE_PSNR:.1f} dB)')
for lam in LAMBDA_VALUES:
    r = ffl_results[lam]
    axes[1].plot(r['psnr'], label=r['label'], lw=2)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('PSNR (dB)')
axes[1].set_title('FFL — PSNR Curves', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# Bar chart
all_labels = [f'A2\nBaseline'] + [f'λ={l}' for l in LAMBDA_VALUES]
all_psnrs  = [A2_BASELINE_PSNR] + [ffl_results[l]['final_psnr'] for l in LAMBDA_VALUES]
bar_colors = ['steelblue', 'blue', 'red', 'cyan']
bars = axes[2].bar(all_labels, all_psnrs, color=bar_colors, edgecolor='black', lw=0.8)
for b, v in zip(bars, all_psnrs):
    axes[2].text(b.get_x()+b.get_width()/2, b.get_height()+0.2,
                 f'{v:.2f}', ha='center', fontweight='bold')
axes[2].set_ylabel('Final PSNR (dB)')
axes[2].set_title('FFL — PSNR vs Lambda', fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Improvement II — Fourier Frequency Loss (Valeena Afzal)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp2_ffl.png')

print('\nFFL Summary:')
print(f'  A2 Baseline : {A2_BASELINE_PSNR:.2f} dB')
for lam in LAMBDA_VALUES:
    r = ffl_results[lam]
    delta = r['final_psnr'] - A2_BASELINE_PSNR
    print(f'  λ={lam:<4} : {r["final_psnr"]:.2f} dB  (Δ {delta:.2f} dB) — Final loss {r["losses_total"][-1]:.4f}')

---
# Section 4 — Improvement III: H-SIREN + Poisson (Maryam Zafar)
**H-SIREN** replaces the standard first-layer activation `sin(ω₀x)` with `sin(sinh(2x))`  
(Gao & Jaiman, arXiv:2410.04716, 2024).

Two Poisson image-reconstruction experiments:  
1. **Cameraman** — same image as A2.  
2. **Coffee** — second test image from skimage.

Loss: gradient supervision  `L = ‖∇Φ(x) - ∇f(x)‖²`

In [ ]:
class PoissonEqn(Dataset):
    def __init__(self, sidelength, image='cameraman'):
        super().__init__()
        if image == 'cameraman':
            img = get_cameraman_tensor(sidelength)
        else:  # coffee
            raw = skimage.data.coffee()
            raw_gray = np.mean(raw, axis=2).astype(np.float32)  # convert RGB→gray
            pil = Image.fromarray((raw_gray).astype(np.uint8))
            transform = Compose([
                Resize(sidelength),
                ToTensor(),
                Normalize(torch.Tensor([0.5]), torch.Tensor([0.5]))
            ])
            img = transform(pil)

        grads_x = scipy.ndimage.sobel(img.numpy(), axis=1).squeeze(0)[..., None]
        grads_y = scipy.ndimage.sobel(img.numpy(), axis=2).squeeze(0)[..., None]
        grads_x = torch.from_numpy(grads_x)
        grads_y = torch.from_numpy(grads_y)
        self.grads   = torch.stack((grads_x, grads_y), dim=-1).view(-1, 2)
        self.laplace_ = scipy.ndimage.laplace(img.numpy()).squeeze(0)[..., None]
        self.laplace_ = torch.from_numpy(self.laplace_)
        self.pixels  = img.permute(1, 2, 0).view(-1, 1)
        self.coords  = get_mgrid(sidelength, 2)

    def __len__(self): return 1

    def __getitem__(self, idx):
        return self.coords, {'pixels': self.pixels, 'grads': self.grads,
                              'laplace': self.laplace_}


def train_poisson(model_class, sidelength=128, total_steps=10_000,
                  lr=1e-4, log_every=1000, image='cameraman'):
    ds = PoissonEqn(sidelength, image=image)
    dl = DataLoader(ds, batch_size=1, pin_memory=True, num_workers=0)
    model_input, gt = next(iter(dl))
    gt = {k: v.to(device) for k, v in gt.items()}
    model_input = model_input.to(device)

    model = model_class(in_features=2, hidden_features=256, hidden_layers=4,
                         out_features=1, outermost_linear=True).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr)

    losses = []
    t0 = time.time()
    for step in range(total_steps):
        model_output, coords = model(model_input)
        grads    = gradient(model_output, coords)
        train_loss = torch.mean((grads - gt['grads']) ** 2 * grads.shape[-1])
        opt.zero_grad(); train_loss.backward(); opt.step()
        losses.append(train_loss.item())
        if step % log_every == 0:
            print(f'    {step:5d} | Loss {train_loss.item():.4e}')

    elapsed = (time.time() - t0) / 60
    # Compute PSNR against pixels
    with torch.no_grad():
        mo, _ = model(model_input)
    mo_np = mo.cpu().view(sidelength, sidelength).numpy()
    gt_np = gt['pixels'].cpu().view(sidelength, sidelength).numpy()
    # Normalise to [0,1] for PSNR
    mo_norm = (mo_np - mo_np.min()) / (mo_np.max() - mo_np.min() + 1e-8)
    gt_norm = (gt_np - gt_np.min()) / (gt_np.max() - gt_np.min() + 1e-8)
    final_psnr = calc_psnr(gt_norm, mo_norm, data_range=1.)
    mse_val    = float(np.mean((mo_norm - gt_norm)**2))
    print(f'    Done {elapsed:.1f}m | PSNR={final_psnr:.2f} dB | MSE={mse_val:.6f}')
    return {'losses': losses, 'psnr': final_psnr, 'mse': mse_val,
            'pred': mo_norm, 'gt': gt_norm, 'time_min': elapsed}


POISSON_STEPS = 10_000
print('=== Improvement III: H-SIREN + Poisson ===')
print('\n--- Cameraman image ---')
print('  SIREN:')
p_siren_cam  = train_poisson(Siren,  128, POISSON_STEPS, image='cameraman')
print('  H-SIREN:')
p_hsiren_cam = train_poisson(HSiren, 128, POISSON_STEPS, image='cameraman')

print('\n--- Coffee image ---')
print('  SIREN:')
p_siren_cof  = train_poisson(Siren,  128, POISSON_STEPS, image='coffee')
print('  H-SIREN:')
p_hsiren_cof = train_poisson(HSiren, 128, POISSON_STEPS, image='coffee')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(25, 10))
rows = [
    ('Cameraman', p_siren_cam, p_hsiren_cam),
    ('Coffee',    p_siren_cof, p_hsiren_cof),
]
for row_idx, (name, ps, ph) in enumerate(rows):
    diff = np.abs(ps['gt'] - ps['pred'])
    diff_h = np.abs(ph['gt'] - ph['pred'])
    axes[row_idx,0].imshow(ps['gt'],    cmap='gray'); axes[row_idx,0].set_title(f'GT {name}',    fontweight='bold')
    axes[row_idx,1].imshow(ps['pred'],  cmap='gray'); axes[row_idx,1].set_title(f'SIREN\n{ps["psnr"]:.2f} dB',  fontweight='bold')
    axes[row_idx,2].imshow(ph['pred'],  cmap='gray'); axes[row_idx,2].set_title(f'H-SIREN\n{ph["psnr"]:.2f} dB', fontweight='bold')
    axes[row_idx,3].imshow(diff,  cmap='hot');  axes[row_idx,3].set_title('|GT−SIREN|',   fontweight='bold')
    axes[row_idx,4].imshow(diff_h, cmap='hot'); axes[row_idx,4].set_title('|GT−H-SIREN|', fontweight='bold')
    for ax in axes[row_idx]: ax.axis('off')

plt.suptitle('Improvement III — Poisson Reconstruction: SIREN vs H-SIREN (Maryam Zafar)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp3_poisson.png')

# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, (name, ps, ph) in zip(axes, rows):
    ax.semilogy(ps['losses'], label='SIREN',   color='blue',   lw=2)
    ax.semilogy(ph['losses'], label='H-SIREN', color='orange', lw=2)
    ax.set_title(f'Poisson Loss — {name}', fontweight='bold')
    ax.set_xlabel('Step'); ax.set_ylabel('Loss (log)')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.suptitle('Improvement III — Training Loss Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp3_poisson_loss.png')

print('\nPoisson Summary:')
print(f'  Cameraman  — SIREN: {p_siren_cam["psnr"]:.2f} dB | H-SIREN: {p_hsiren_cam["psnr"]:.2f} dB')
print(f'  Coffee     — SIREN: {p_siren_cof["psnr"]:.2f} dB | H-SIREN: {p_hsiren_cof["psnr"]:.2f} dB')

---
# Section 5 — Improvement IV-A: Helmholtz k-Ablation (Laiba Noor)
**Equation:** ∇²u + k²n²u = −q  
**Novel experiment:** k ∈ {1, 5, 20, 50} — the original paper only tested k=1.  
Architecture: 4×256 SIREN, ω₀=30, Adam lr=2e-5, **1 000 steps per k**.  
Key finding: SIREN converges for all k with 100% loss reduction.

In [ ]:
K_VALUES  = [1, 5, 20, 50]
H_STEPS   = 1_000
H_LR      = 2e-5
H_SIDE    = 150
H_LOG     = 100
COLORS_H  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
# A2 reference numbers
A2_H_INIT, A2_H_FINAL = 5_581_349, 175_564


def build_helm_batch(side, k):
    lin    = torch.linspace(-1, 1, side)
    x, y   = torch.meshgrid(lin, lin, indexing='ij')
    coords = torch.stack([x.reshape(-1), y.reshape(-1)], dim=-1)
    dists  = torch.norm(coords, dim=-1)
    source = torch.exp(-dists**2 / (2 * 0.01**2)).unsqueeze(-1)
    vel    = torch.ones(coords.shape[0], 1)
    return coords, source, vel


def train_helmholtz(k):
    print(f'  k={k} ...')
    c_cpu, src, vel = build_helm_batch(H_SIDE, k)
    src = src.to(device); vel = vel.to(device)

    # 4×256 SIREN, output 2 channels (real + imaginary)
    model = Siren(in_features=2, hidden_features=256, hidden_layers=3,
                  out_features=2, outermost_linear=True).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=H_LR)

    losses = []
    t0 = time.time()
    for step in range(H_STEPS):
        model.train()
        c = c_cpu.to(device).requires_grad_(True)
        u = model.forward_simple(c)
        laps = []
        for ch in range(2):
            g = torch.autograd.grad(u[:, ch:ch+1], c,
                    grad_outputs=torch.ones_like(u[:, ch:ch+1]),
                    create_graph=True)[0]
            lap = sum(
                torch.autograd.grad(g[:, d:d+1], c,
                    grad_outputs=torch.ones_like(g[:, d:d+1]),
                    create_graph=True)[0][:, d:d+1]
                for d in range(2)
            )
            laps.append(lap)
        k2n2 = k**2 * vel**2
        loss  = ((laps[0] + k2n2*u[:, 0:1] + src)**2).sum() + \
                ((laps[1] + k2n2*u[:, 1:2])**2).sum()
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step % H_LOG == 0:
            print(f'    {step:4d} | {losses[-1]:>16,.0f}')

    elapsed = (time.time() - t0) / 60
    model.eval()
    with torch.no_grad():
        wf = model.forward_simple(c_cpu.to(device)).cpu().numpy()
    red = (1 - losses[-1] / losses[0]) * 100
    # Steps to 95% reduction
    norm = np.array(losses) / losses[0]
    idx  = np.where(norm <= 0.05)[0]
    steps95 = int(idx[0]) if len(idx) else H_STEPS
    print(f'  Done {elapsed:.1f}m | {losses[0]:.0f}→{losses[-1]:.0f} ({red:.1f}%) | Steps95={steps95}')
    return {'losses': losses, 'init': losses[0], 'final': losses[-1],
            'reduction': red, 'steps95': steps95, 'time_min': elapsed,
            'wf': wf, 'side': H_SIDE}


print('=== Improvement IV-A: Helmholtz k-Ablation ===')
helm = {k: train_helmholtz(k) for k in K_VALUES}
print('Done!')

In [ ]:
# Figure 1 — Loss Curves / Final Loss / Reduction %
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
for i, k in enumerate(K_VALUES):
    ax.semilogy(helm[k]['losses'], color=COLORS_H[i], label=f'k={k}', lw=2.2)
ax.axhline(A2_H_FINAL, color='gray', ls='--', lw=1.5, label='A2 final (5K steps)')
ax.set_xlabel('Step'); ax.set_ylabel('Loss (log)')
ax.set_title('Helmholtz Loss Curves — k-Ablation', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
finals = [helm[k]['final'] for k in K_VALUES]
bars   = ax2.bar([f'k={k}' for k in K_VALUES], finals, color=COLORS_H, edgecolor='black', lw=0.8)
for b, v in zip(bars, finals):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()*1.03,
             f'{v:.0f}', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('Final Loss'); ax2.set_title('Final Loss vs k', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

ax3 = axes[2]
reds = [helm[k]['reduction'] for k in K_VALUES]
bars3 = ax3.bar([f'k={k}' for k in K_VALUES], reds, color=COLORS_H, edgecolor='black', lw=0.8)
for b, v in zip(bars3, reds):
    ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.5,
             f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')
ax3.set_ylabel('Loss Reduction (%)'); ax3.set_ylim(0, 110)
ax3.set_title('Reduction % vs k', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('FIGURE 1 — Helmholtz k-Ablation: Loss / Reduction (Laiba Noor)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp4a_fig1_helmholtz_loss.png')

In [ ]:
# Figure 2 — Wavefields (Real / Imaginary / Amplitude)
S   = H_SIDE
fig, axes = plt.subplots(3, 4, figsize=(22, 15))
row_labels = ['Real part  u_r', 'Imaginary part  u_i', 'Amplitude  |u|']

for col, k in enumerate(K_VALUES):
    wf    = helm[k]['wf']
    r_ch  = wf[:, 0].reshape(S, S)
    i_ch  = wf[:, 1].reshape(S, S)
    amp   = np.sqrt(r_ch**2 + i_ch**2)
    vr    = np.percentile(np.abs(r_ch), 99) or 1
    vi    = np.percentile(np.abs(i_ch), 99) or 1

    i0 = axes[0, col].imshow(r_ch,  cmap='seismic', origin='lower', vmin=-vr, vmax=vr)
    axes[0, col].set_title(f'k={k}\n{row_labels[0]}', fontweight='bold'); axes[0, col].axis('off')
    plt.colorbar(i0, ax=axes[0, col], fraction=0.046)

    i1 = axes[1, col].imshow(i_ch,  cmap='seismic', origin='lower', vmin=-vi, vmax=vi)
    axes[1, col].set_title(f'k={k}\n{row_labels[1]}', fontweight='bold'); axes[1, col].axis('off')
    plt.colorbar(i1, ax=axes[1, col], fraction=0.046)

    i2 = axes[2, col].imshow(amp,   cmap='inferno',  origin='lower')
    axes[2, col].set_title(f'k={k}\n{row_labels[2]}', fontweight='bold'); axes[2, col].axis('off')
    plt.colorbar(i2, ax=axes[2, col], fraction=0.046)

plt.suptitle('FIGURE 2 — Helmholtz Wavefield: Real / Imaginary / Amplitude per k',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('imp4a_fig2_wavefields.png')

In [ ]:
# Figure 3 — Convergence Speed + Table
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for i, k in enumerate(K_VALUES):
    ls = np.array(helm[k]['losses'])
    ax.plot(ls / ls[0], color=COLORS_H[i], label=f'k={k}', lw=2.2)
ax.axhline(0.05, color='black', ls=':', lw=1.5, label='95% reduction')
ax.set_xlabel('Step'); ax.set_ylabel('Normalised Loss')
ax.set_title('Convergence Speed — Normalised Loss', fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3)

ax2 = axes[1]
steps95 = [helm[k]['steps95'] for k in K_VALUES]
bars = ax2.bar([f'k={k}' for k in K_VALUES], steps95, color=COLORS_H, edgecolor='black', lw=0.8)
for b, v in zip(bars, steps95):
    lbl = str(v) if v < H_STEPS else f'>{H_STEPS}'
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()+1,
             lbl, ha='center', fontweight='bold')
ax2.set_ylabel('Steps to 95% Reduction')
ax2.set_title('Convergence Speed per k', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('FIGURE 3 — Helmholtz Convergence Speed Analysis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp4a_fig3_convergence.png')

# Print table
print('\n' + '─'*80)
print(f'{"Table 1 — Helmholtz k-Ablation":^80}')
print('─'*80)
print(f'{"Config":<25} {"k":>4} {"Init Loss":>14} {"Final Loss":>12} {"Reduction":>10} {"Steps95":>8}')
print('─'*80)
print(f'{"A2 baseline (5K steps)":<25} {1:>4} {A2_H_INIT:>14,} {A2_H_FINAL:>12,} {"96.9%":>10} {"—":>8}')
print('─'*80)
for k in K_VALUES:
    r = helm[k]
    tag = 'repro' if k == 1 else 'new'
    print(f'{f"SIREN k={k} [{tag}]":<25} {k:>4} {r["init"]:>14,.0f} {r["final"]:>12,.0f} '
          f'{r["reduction"]:>9.1f}% {r["steps95"]:>8}')
print('─'*80)

---
# Section 6 — Improvement IV-B: Wave Equation IC Study (Laiba Noor)
**Equation:** ∂²u/∂t² = c²(∂²u/∂x² + ∂²u/∂y²)  
3D input (x, y, t), scalar output u(x,t).

Three initial conditions tested:  
- **IC-1** (A2 default): u(x,0) = sin(πx)  
- **IC-2** (new): u(x,0) = exp(−‖x‖²/0.05)  — Gaussian pulse  
- **IC-3** (new): u(x,0) = sin(2πx) + 0.5·sin(4πx)  — Double sinusoid

In [ ]:
W_STEPS  = 500
W_LR     = 2e-5
W_BATCH  = 3_000
W_LOG    = 50
A2_W_INIT, A2_W_FINAL = 85_955_296, 16_236

IC_TYPES  = ['default', 'gaussian', 'double_sin']
IC_LABELS = {
    'default':    'IC-1: sin(πx) [A2]',
    'gaussian':   'IC-2: Gaussian [NEW]',
    'double_sin': 'IC-3: Double-sin [NEW]'
}
IC_COLORS = ['#1f77b4', '#2ca02c', '#d62728']


def ic_val(ic, x, y):
    if ic == 'default':    return torch.sin(np.pi * x)
    elif ic == 'gaussian': return torch.exp(-(x**2 + y**2) / 0.05)
    else:                  return torch.sin(2*np.pi*x) + 0.5*torch.sin(4*np.pi*x)


def train_wave(ic):
    print(f'  IC={ic} ...')
    model = Siren(in_features=3, hidden_features=256, hidden_layers=3,
                  out_features=1, outermost_linear=True).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=W_LR)

    losses = []
    t0 = time.time()
    for step in range(W_STEPS):
        model.train()
        coords = (torch.rand(W_BATCH, 3)*2 - 1).to(device).requires_grad_(True)
        u  = model.forward_simple(coords)
        du = torch.autograd.grad(u, coords,
                                  grad_outputs=torch.ones_like(u),
                                  create_graph=True)[0]
        d2t = torch.autograd.grad(du[:, 2:3], coords,
                                   grad_outputs=torch.ones_like(du[:, 2:3]),
                                   create_graph=True)[0][:, 2:3]
        d2x = torch.autograd.grad(du[:, 0:1], coords,
                                   grad_outputs=torch.ones_like(du[:, 0:1]),
                                   create_graph=True)[0][:, 0:1]
        d2y = torch.autograd.grad(du[:, 1:2], coords,
                                   grad_outputs=torch.ones_like(du[:, 1:2]),
                                   create_graph=True)[0][:, 1:2]
        pde_loss = ((d2t - d2x - d2y)**2).sum()

        # Initial condition
        n_ic  = W_BATCH // 5
        xy_ic = (torch.rand(n_ic, 2)*2 - 1).to(device)
        c_ic  = torch.cat([xy_ic, torch.zeros(n_ic, 1).to(device)], dim=-1).requires_grad_(True)
        u_ic  = model.forward_simple(c_ic)
        gt_ic = ic_val(ic, xy_ic[:, 0], xy_ic[:, 1]).unsqueeze(-1).to(device)
        ic_loss  = ((u_ic - gt_ic)**2).sum()
        du_ic    = torch.autograd.grad(u_ic, c_ic,
                                        grad_outputs=torch.ones_like(u_ic),
                                        create_graph=True)[0]
        vel_loss = (du_ic[:, 2:3]**2).sum()

        loss = pde_loss + ic_loss + vel_loss
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
        if step % W_LOG == 0:
            print(f'    {step:4d} | {losses[-1]:>18,.0f}')

    elapsed = (time.time() - t0) / 60
    red = (1 - losses[-1] / losses[0]) * 100
    print(f'  Done {elapsed:.1f}m | {losses[0]:.0f}→{losses[-1]:.0f} ({red:.1f}%)')

    # Spacetime visualisation (y=0 slice)
    lin = torch.linspace(-1, 1, 80)
    tv  = torch.linspace(0, 1, 80)
    gx, gt_t = torch.meshgrid(lin, tv, indexing='ij')
    vc  = torch.stack([gx.reshape(-1), torch.zeros_like(gx).reshape(-1),
                       gt_t.reshape(-1)], dim=-1)
    model.eval()
    with torch.no_grad():
        u_vis = model.forward_simple(vc.to(device)).cpu().numpy().reshape(80, 80)

    return {'losses': losses, 'init': losses[0], 'final': losses[-1],
            'reduction': red, 'time_min': elapsed, 'u_vis': u_vis}


print('=== Improvement IV-B: Wave Equation IC Variation ===')
wave = {ic: train_wave(ic) for ic in IC_TYPES}
print('Done!')

In [ ]:
# Figure 4 — Loss / Final Loss / Normalised Convergence
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

ax = axes[0]
for i, ic in enumerate(IC_TYPES):
    ax.semilogy(wave[ic]['losses'], color=IC_COLORS[i], label=IC_LABELS[ic], lw=2.2)
ax.axhline(A2_W_FINAL, color='gray', ls='--', lw=1.5, label='A2 final (5K steps)')
ax.set_xlabel('Step'); ax.set_ylabel('Loss (log)')
ax.set_title('Wave Equation Loss — IC Variation', fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

ax2 = axes[1]
wfin  = [wave[ic]['final'] for ic in IC_TYPES]
short = ['IC-1\nsin(πx)', 'IC-2\nGaussian', 'IC-3\nDouble-sin']
bars  = ax2.bar(short, wfin, color=IC_COLORS, edgecolor='black', lw=0.8)
for b, v in zip(bars, wfin):
    ax2.text(b.get_x()+b.get_width()/2, b.get_height()*1.03,
             f'{v:.0f}', ha='center', va='bottom', fontweight='bold')
ax2.set_ylabel('Final Loss at 500 steps')
ax2.set_title('Final Loss by IC', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

ax3 = axes[2]
for i, ic in enumerate(IC_TYPES):
    ls = np.array(wave[ic]['losses'])
    ax3.plot(ls / ls[0], color=IC_COLORS[i], label=IC_LABELS[ic], lw=2.2)
ax3.axhline(0.05, color='black', ls=':', lw=1.5, label='95% reduction')
ax3.set_xlabel('Step'); ax3.set_ylabel('Normalised Loss')
ax3.set_title('Convergence Speed — Normalised', fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

plt.suptitle('FIGURE 4 — Wave Equation IC Variation (Improvement IV-B, Laiba Noor)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp4b_fig4_wave_loss.png')

In [ ]:
# Figure 5 — Spacetime Solutions u(x,t)
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for col, ic in enumerate(IC_TYPES):
    uv  = wave[ic]['u_vis']
    vm  = np.percentile(np.abs(uv), 99) or 1

    im1 = axes[0, col].imshow(uv, cmap='RdBu', origin='lower', vmin=-vm, vmax=vm,
                               aspect='auto', extent=[-1, 1, 0, 1])
    axes[0, col].set_title(f'{IC_LABELS[ic]}\nu(x,t)', fontweight='bold')
    axes[0, col].set_xlabel('x'); axes[0, col].set_ylabel('t')
    plt.colorbar(im1, ax=axes[0, col], fraction=0.046)

    im2 = axes[1, col].imshow(np.abs(uv), cmap='hot', origin='lower',
                               aspect='auto', extent=[-1, 1, 0, 1])
    axes[1, col].set_title(f'{IC_LABELS[ic]}\n|u(x,t)| Amplitude', fontweight='bold')
    axes[1, col].set_xlabel('x'); axes[1, col].set_ylabel('t')
    plt.colorbar(im2, ax=axes[1, col], fraction=0.046)

plt.suptitle('FIGURE 5 — Wave Spacetime Solutions u(x,t) per Initial Condition',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('imp4b_fig5_wave_spacetime.png')

In [ ]:
# Figure 6 — IC Profiles
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
xv  = torch.linspace(-1, 1, 300)
yv  = torch.zeros(300)
formulas = {
    'default':    'sin(πx)',
    'gaussian':   'exp(−‖x‖²/0.05)',
    'double_sin': 'sin(2πx)+0.5·sin(4πx)'
}
for col, ic in enumerate(IC_TYPES):
    vals = ic_val(ic, xv, yv).numpy()
    axes[col].plot(xv.numpy(), vals, color=IC_COLORS[col], lw=2.5)
    axes[col].fill_between(xv.numpy(), 0, vals, alpha=0.2, color=IC_COLORS[col])
    axes[col].set_title(f'{IC_LABELS[ic]}\nu(x,0)={formulas[ic]}', fontweight='bold')
    axes[col].set_xlabel('x'); axes[col].set_ylabel('u(x,0)')
    axes[col].axhline(0, color='gray', lw=0.8)
    axes[col].grid(True, alpha=0.3)

plt.suptitle('FIGURE 6 — Initial Condition Profiles u(x,y=0,t=0)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig('imp4b_fig6_ic_profiles.png')

# Table
print('\n' + '─'*80)
print(f'{"Table 2 — Wave Equation IC Variation":^80}')
print('─'*80)
print(f'{"Experiment":<36} {"Init Loss":>14} {"Final Loss":>11} {"Reduction":>10} {"Steps":>6}')
print('─'*80)
print(f'{"A2 (IC-1, default, 5K steps)":<36} {A2_W_INIT:>14,} {A2_W_FINAL:>11,} {"99.98%":>10} {"5000":>6}')
print('─'*80)
for ic in IC_TYPES:
    r = wave[ic]
    print(f'{IC_LABELS[ic]:<36} {r["init"]:>14,.0f} {r["final"]:>11,.0f} {r["reduction"]:>9.1f}% {W_STEPS:>6}')
print('─'*80)

---
# Section 7 — Bonus: Brain MRI Super-Resolution (Laiba Noor)
**Task:** (x,y) → pixel intensity.  Train SIREN on LR coordinates, evaluate at HR resolution.  
**Dataset:** IXI T1 brain MRI (5 sagittal slices) — falls back to synthetic slices if unavailable.  
**Scales:** 2× and 4× downsampling.  
**Methods:** Bicubic, ReLU MLP, SIREN (A2), SIREN+FFL+CAOS (A3 combined).

In [ ]:
TARGET_SZ  = 128
NUM_SLICES = 5
SCALES_MRI = [2, 4]
MRI_DIR    = '/content/ixi_mri'
os.makedirs(MRI_DIR, exist_ok=True)

hr_slices = []
try:
    import torchio as tio
    ixi = tio.datasets.IXI(root=MRI_DIR, modalities=['T1'], download=True)
    for subj in ixi:
        if len(hr_slices) >= NUM_SLICES: break
        try:
            vol = subj['T1'][tio.DATA].squeeze().numpy()
            sl  = vol[:, :, vol.shape[2]//2].astype(np.float32)
            sl  = sk_resize(sl, (TARGET_SZ, TARGET_SZ),
                            anti_aliasing=True, preserve_range=True)
            sl  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
            hr_slices.append(sl)
        except Exception: pass
    print(f'IXI loaded: {len(hr_slices)} slices.')
except Exception as e:
    print(f'IXI unavailable ({e}). Using synthetic slices.')

if len(hr_slices) < NUM_SLICES:
    print('Generating synthetic brain-like slices...')
    np.random.seed(42)
    while len(hr_slices) < NUM_SLICES:
        sl  = np.zeros((TARGET_SZ, TARGET_SZ), np.float32)
        y, x = np.ogrid[-1:1:TARGET_SZ*1j, -1:1:TARGET_SZ*1j]
        brain = (x**2/0.7**2 + y**2/0.85**2) < 1.0
        grey  = (x**2/0.5**2 + y**2/0.60**2) < 1.0
        sl[brain] = 0.30 + 0.08*np.random.randn(TARGET_SZ, TARGET_SZ)[brain]
        sl[grey & brain] += 0.35 + 0.06*np.random.randn(TARGET_SZ, TARGET_SZ)[grey & brain]
        sl = np.clip(sl + 0.04*np.random.randn(TARGET_SZ, TARGET_SZ)*brain, 0, 1)
        hr_slices.append(sl)
    print(f'Generated {len(hr_slices)} synthetic slices.')

def downsample(img, s):
    return sk_resize(img, (img.shape[0]//s, img.shape[1]//s),
                     anti_aliasing=True, preserve_range=True)

def bicubic_up(lr, sz):
    return sk_resize(lr, (sz, sz), order=3,
                     anti_aliasing=True, preserve_range=True)

print(f'{len(hr_slices)} HR slices ({TARGET_SZ}×{TARGET_SZ}) ready.')

In [ ]:
MRI_STEPS = 400
MRI_LR    = 1e-4
MRI_LOG   = 100


def caos_omega_mri(step, total, lo=10., hi=60.):
    return lo + (hi - lo) * (1 - np.cos(np.pi * step / total)) / 2


def train_mri(lr_img, hr_img, mtype='siren', use_ffl=False, use_caos=False):
    H  = lr_img.shape[0]
    Hr = hr_img.shape[0]
    if mtype == 'siren':
        model = Siren(in_features=2, hidden_features=256, hidden_layers=5,
                      out_features=1, outermost_linear=True).to(device)
    else:
        model = ReLUMLP(in_features=2, hidden_features=256,
                        hidden_layers=5, out_features=1).to(device)
    opt  = torch.optim.Adam(model.parameters(), lr=MRI_LR)
    clr  = make_grid_2d(H).to(device)
    plr  = torch.from_numpy(lr_img.reshape(-1, 1)).float().to(device)
    chr_ = make_grid_2d(Hr).to(device)

    losses = []
    for step in range(MRI_STEPS):
        model.train()
        # CAOS: update ω₀ every 40 steps
        if use_caos and mtype == 'siren' and step % 40 == 0:
            w = caos_omega_mri(step, MRI_STEPS)
            for m in model.net.modules():
                if isinstance(m, SineLayer):
                    m.omega_0 = w

        if mtype == 'siren':
            pred = model.forward_simple(clr)
        else:
            pred = model(clr)

        l2   = ((pred - plr)**2).mean()
        loss = l2
        if use_ffl:
            lam   = 0.05 * min(1., step / 200)
            p2d   = pred.reshape(H, H)
            g2d   = plr.reshape(H, H)
            ffl   = torch.mean(torch.abs(
                        torch.abs(torch.fft.fft2(p2d)) -
                        torch.abs(torch.fft.fft2(g2d))))
            loss  = l2 + lam * ffl

        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        if mtype == 'siren':
            ph = model.forward_simple(chr_).cpu().numpy().reshape(Hr, Hr)
        else:
            ph = model(chr_).cpu().numpy().reshape(Hr, Hr)
    ph = np.clip(ph, 0, 1)
    return {
        'psnr':   calc_psnr(hr_img, ph, data_range=1.),
        'ssim':   calc_ssim(hr_img, ph, data_range=1.),
        'pred':   ph,
        'losses': losses
    }


METHODS = [
    ('bicubic',       'Bicubic',          False, False),
    ('relu',          'ReLU MLP',         False, False),
    ('siren',         'SIREN (A2)',        False, False),
    ('siren_ffl_caos','SIREN+FFL+CAOS',   True,  True),
]
MC = ['#aec7e8', '#ffbb78', '#98df8a', '#ff9896']

mri_res = {s: {m[0]: [] for m in METHODS} for s in SCALES_MRI}
mri_ex  = {}

print('=== Bonus: Brain MRI Super-Resolution ===')
for scale in SCALES_MRI:
    print(f'\n--- Scale {scale}× ---')
    for si, hr in enumerate(hr_slices):
        lr = downsample(hr, scale)
        print(f'  Slice {si+1} | LR={lr.shape}')
        for mkey, mname, ffl, caos in METHODS:
            if mkey == 'bicubic':
                bic = bicubic_up(lr, TARGET_SZ)
                res = {'psnr': calc_psnr(hr, bic, data_range=1.),
                       'ssim': calc_ssim(hr, bic, data_range=1.),
                       'pred': bic, 'losses': []}
            else:
                mtype = 'relu' if mkey == 'relu' else 'siren'
                res   = train_mri(lr, hr, mtype=mtype, use_ffl=ffl, use_caos=caos)
            mri_res[scale][mkey].append(res)
            print(f'    {mname:<28} PSNR={res["psnr"]:6.2f}  SSIM={res["ssim"]:.4f}')
        if si == 0:
            mri_ex[(scale, 'hr')] = hr
            mri_ex[(scale, 'lr')] = lr
            for mkey, _, _, _ in METHODS:
                mri_ex[(scale, mkey)] = mri_res[scale][mkey][0]['pred']

print('\n✓ MRI experiments complete!')

In [ ]:
# Figure 7 — Visual Comparison
order  = ['hr', 'lr', 'bicubic', 'relu', 'siren', 'siren_ffl_caos']
titles = {'hr':            'HR Ground Truth',
           'lr':            'LR Input\n(upscaled)',
           'bicubic':       'Bicubic',
           'relu':          'ReLU MLP',
           'siren':         'SIREN (A2)',
           'siren_ffl_caos':'SIREN+FFL\n+CAOS (A3)'}

fig, axes = plt.subplots(2, 6, figsize=(24, 9))
for row, scale in enumerate(SCALES_MRI):
    for col, mk in enumerate(order):
        ax  = axes[row, col]
        img = mri_ex.get((scale, mk))
        if img is None: ax.axis('off'); continue
        show = bicubic_up(img, TARGET_SZ) if mk == 'lr' else img
        ax.imshow(show, cmap='gray', vmin=0, vmax=1)
        t = titles[mk]
        if mk not in ('hr', 'lr'):
            d = mri_res[scale][mk][0]
            t += f'\n{d["psnr"]:.2f} dB | {d["ssim"]:.3f}'
        ax.set_title(t, fontsize=9, fontweight='bold')
        ax.axis('off')
        if col == 0: ax.set_ylabel(f'{scale}× SR', fontsize=12, fontweight='bold')

plt.suptitle('FIGURE 7 — MRI Super-Resolution Visual Comparison (Slice 1, IXI T1 Brain)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('bonus_fig7_mri_visual.png')

In [ ]:
# Figure 8 — PSNR & SSIM bars
mkeys  = [m[0] for m in METHODS]
mshort = ['Bicubic', 'ReLU\nMLP', 'SIREN\n(A2)', 'SIREN+\nFFL+CAOS']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for row, scale in enumerate(SCALES_MRI):
    pm = [np.mean([d['psnr'] for d in mri_res[scale][mk]]) for mk in mkeys]
    ps = [np.std ([d['psnr'] for d in mri_res[scale][mk]]) for mk in mkeys]
    sm = [np.mean([d['ssim'] for d in mri_res[scale][mk]]) for mk in mkeys]
    ss = [np.std ([d['ssim'] for d in mri_res[scale][mk]]) for mk in mkeys]

    ax = axes[row, 0]
    bars = ax.bar(mshort, pm, yerr=ps, color=MC, edgecolor='black', lw=0.8, capsize=6)
    for b, v in zip(bars, pm):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                f'{v:.2f}', ha='center', fontweight='bold')
    ax.set_ylabel('PSNR (dB)'); ax.set_title(f'PSNR — {scale}× SR', fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    ax2 = axes[row, 1]
    bars2 = ax2.bar(mshort, sm, yerr=ss, color=MC, edgecolor='black', lw=0.8, capsize=6)
    for b, v in zip(bars2, sm):
        ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.005,
                 f'{v:.4f}', ha='center', fontweight='bold')
    ax2.set_ylabel('SSIM'); ax2.set_ylim(0, 1.1)
    ax2.set_title(f'SSIM — {scale}× SR', fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('FIGURE 8 — MRI SR: PSNR & SSIM (Mean ± Std, 5 slices)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
save_fig('bonus_fig8_mri_psnr_ssim.png')

# Table
print('\n' + '─'*75)
print(f'{"Table 3 — MRI Super-Resolution Results (Bonus)":^75}')
print('─'*75)
for scale in SCALES_MRI:
    print(f'\n  {scale}× Upsampling')
    print(f'  {"Method":<28} {"Mean PSNR":>12} {"Mean SSIM":>12}')
    print(f'  {"-"*55}')
    for mk, mname, _, _ in METHODS:
        p = np.mean([d['psnr'] for d in mri_res[scale][mk]])
        s = np.mean([d['ssim'] for d in mri_res[scale][mk]])
        print(f'  {mname:<28} {p:>11.2f} {s:>12.4f}')
print('─'*75)

---
# Section 8 — Consolidated Summary & CSV Export

In [ ]:
print('=' * 80)
print('  SIREN A3 — COMPLETE RESULTS SUMMARY')
print('  Group ID09 | FAST-NUCES | Dr. Zohair Ahmed | April 2026')
print('=' * 80)

print(f"""
┌────────────────────────────────────────────────────────────────────────────┐
│  A2 Baseline — Image Fitting                                               │
│  SIREN (fixed ω₀=30, 10K steps): PSNR = {A2_BASELINE_PSNR:.2f} dB                      │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│  Improvement I — CAOS-SIREN (Valeena)                                      │
│  ω₀ 25→45 (narrow): PSNR = {caos_narrow['final_psnr']:.2f} dB  (Δ {caos_narrow['final_psnr']-A2_BASELINE_PSNR:+.2f} dB)                  │
│  ω₀ 10→60 (wide)  : PSNR = {caos_wide['final_psnr']:.2f} dB  (Δ {caos_wide['final_psnr']-A2_BASELINE_PSNR:+.2f} dB)                  │
│  → Dynamic ω₀ scheduling destabilises SIREN init. Both runs FAIL.         │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│  Improvement II — Fourier Frequency Loss (Valeena)                         │
│  λ=0.01: PSNR = {ffl_results[0.01]['final_psnr']:.2f} dB  |  λ=0.05: {ffl_results[0.05]['final_psnr']:.2f} dB  |  λ=0.10: {ffl_results[0.1]['final_psnr']:.2f} dB │
│  → FFT loss conflicts with SIREN natural freq learning. All FAIL.          │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│  Improvement III — H-SIREN + Poisson (Maryam)                              │
│  Cameraman: SIREN {p_siren_cam['psnr']:.2f} dB | H-SIREN {p_hsiren_cam['psnr']:.2f} dB  (near-draw)         │
│  Coffee   : SIREN {p_siren_cof['psnr']:.2f} dB | H-SIREN {p_hsiren_cof['psnr']:.2f} dB  (H-SIREN diverged)      │
│  → H-SIREN matches on simple images; diverges on complex tasks.            │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│  Improvement IV-A — Helmholtz k-Ablation (Laiba)                           │
│  k=1: {helm[1]['reduction']:.1f}% | k=5: {helm[5]['reduction']:.1f}% | k=20: {helm[20]['reduction']:.1f}% | k=50: {helm[50]['reduction']:.1f}%                    │
│  → SIREN converges for ALL k values. Extends paper's single-k result.      │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│  Improvement IV-B — Wave IC Variation (Laiba)                              │
│  IC-1 sin(πx): {wave['default']['reduction']:.1f}% | IC-2 Gaussian: {wave['gaussian']['reduction']:.1f}% | IC-3 Double-sin: {wave['double_sin']['reduction']:.1f}%   │
│  → SIREN converges for all 3 ICs. Gaussian fastest (best phys alignment).  │
└────────────────────────────────────────────────────────────────────────────┘
""")

siren_2x_psnr = np.mean([d['psnr'] for d in mri_res[2]['siren']])
bic_2x_psnr   = np.mean([d['psnr'] for d in mri_res[2]['bicubic']])
relu_2x_psnr  = np.mean([d['psnr'] for d in mri_res[2]['relu']])
fflc_2x_psnr  = np.mean([d['psnr'] for d in mri_res[2]['siren_ffl_caos']])

print(f"""┌────────────────────────────────────────────────────────────────────────────┐
│  Bonus — MRI Super-Resolution (Laiba)                                      │
│  2× SR: Bicubic {bic_2x_psnr:.2f} dB | ReLU {relu_2x_psnr:.2f} dB | SIREN {siren_2x_psnr:.2f} dB | FFL+CAOS {fflc_2x_psnr:.2f} dB  │
│  → SIREN within 0.66 dB of bicubic. FFL+CAOS fails (domain mismatch).     │
└────────────────────────────────────────────────────────────────────────────┘
""")

In [ ]:
# ── CSV exports ──
with open(os.path.join(DRIVE_ROOT, 'results_helmholtz.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['k', 'init_loss', 'final_loss', 'reduction_pct', 'steps95', 'steps', 'tag'])
    w.writerow([1, A2_H_INIT, A2_H_FINAL, 96.9, '—', 5000, 'A2_baseline'])
    for k in K_VALUES:
        r = helm[k]
        tag = 'A3_repro' if k == 1 else 'A3_new'
        w.writerow([k, f'{r["init"]:.0f}', f'{r["final"]:.0f}',
                    f'{r["reduction"]:.1f}', r['steps95'], H_STEPS, tag])

with open(os.path.join(DRIVE_ROOT, 'results_wave.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['ic_type', 'init_loss', 'final_loss', 'reduction_pct', 'steps'])
    w.writerow(['A2_default', A2_W_INIT, A2_W_FINAL, 99.98, 5000])
    for ic in IC_TYPES:
        r = wave[ic]
        w.writerow([ic, f'{r["init"]:.0f}', f'{r["final"]:.0f}',
                    f'{r["reduction"]:.1f}', W_STEPS])

with open(os.path.join(DRIVE_ROOT, 'results_image.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['model', 'config', 'final_psnr', 'vs_baseline_db'])
    w.writerow(['A2_Baseline',   'fixed ω₀=30', f'{A2_BASELINE_PSNR:.2f}', '—'])
    w.writerow(['CAOS_narrow',   'ω₀ 25→45',  f'{caos_narrow["final_psnr"]:.2f}',
                f'{caos_narrow["final_psnr"]-A2_BASELINE_PSNR:.2f}'])
    w.writerow(['CAOS_wide',     'ω₀ 10→60',  f'{caos_wide["final_psnr"]:.2f}',
                f'{caos_wide["final_psnr"]-A2_BASELINE_PSNR:.2f}'])
    for lam in LAMBDA_VALUES:
        r = ffl_results[lam]
        w.writerow([f'FFL_lambda={lam}', f'L2+{lam}*FFL',
                    f'{r["final_psnr"]:.2f}',
                    f'{r["final_psnr"]-A2_BASELINE_PSNR:.2f}'])

with open(os.path.join(DRIVE_ROOT, 'results_mri.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['scale', 'method', 'slice_idx', 'psnr', 'ssim'])
    for sc in SCALES_MRI:
        for mk, _, _, _ in METHODS:
            for i, d in enumerate(mri_res[sc][mk]):
                w.writerow([f'{sc}x', mk, i, f'{d["psnr"]:.4f}', f'{d["ssim"]:.6f}'])

# ── Figure checklist ──
FIGS = [
    'a2_baseline_image.png',
    'imp1_caos_siren.png',
    'imp2_ffl.png',
    'imp3_poisson.png',
    'imp3_poisson_loss.png',
    'imp4a_fig1_helmholtz_loss.png',
    'imp4a_fig2_wavefields.png',
    'imp4a_fig3_convergence.png',
    'imp4b_fig4_wave_loss.png',
    'imp4b_fig5_wave_spacetime.png',
    'imp4b_fig6_ic_profiles.png',
    'bonus_fig7_mri_visual.png',
    'bonus_fig8_mri_psnr_ssim.png',
]

print('='*70)
print('  FIGURE CHECKLIST')
print('='*70)
for fg in FIGS:
    ok = '✓' if os.path.exists(os.path.join(DRIVE_ROOT, fg)) else '✗ MISSING'
    print(f'  {ok}  {fg}')
print('\nCSVs:')
for csv_name in ['results_helmholtz.csv', 'results_wave.csv',
                  'results_image.csv', 'results_mri.csv']:
    ok = '✓' if os.path.exists(os.path.join(DRIVE_ROOT, csv_name)) else '✗ MISSING'
    print(f'  {ok}  {csv_name}')
print(f'\nAll outputs in: {DRIVE_ROOT}')
print('='*70)